# Assignment, transportation & network flow

These solvers take **tidy edge lists** — one row per allowed pairing — so sparse problems are natural (just omit rows). Each returns a `SolveResult`.

In [ ]:
import pandas as pd

import ortidy

## Linear assignment

One row per `(agent, task)` pair with its cost. Adds a `selected` column.

In [ ]:
edges = pd.DataFrame({
    "agent": ["a0", "a0", "a0", "a1", "a1", "a1", "a2", "a2", "a2"],
    "task":  ["t0", "t1", "t2", "t0", "t1", "t2", "t0", "t1", "t2"],
    "cost":  [4, 2, 8, 1, 5, 4, 3, 2, 1],
})
result = ortidy.assignment(edges)
print(result.status, "| total cost:", result.objective)
result.frame[result.frame["selected"]]

## Generalized assignment (GAP)

Edges with a `value` and a `size` per `(task, agent)`, plus per-agent capacities. Maximizes value within capacity.

In [ ]:
edges = pd.DataFrame({
    "task":  ["t0", "t0", "t1", "t1", "t2", "t2"],
    "agent": ["a0", "a1", "a0", "a1", "a0", "a1"],
    "value": [10, 6, 8, 9, 5, 7],
    "size":  [3, 2, 4, 3, 2, 4],
})
result = ortidy.generalized_assignment(edges, {"a0": 5, "a1": 6})
print(result.status, "| total value:", result.objective)
result.frame[result.frame["selected"]]

## Transportation

`(source, sink, cost)` lanes plus supply and demand; total supply must equal total demand. Adds a shipped `quantity`.

In [ ]:
edges = pd.DataFrame({
    "source": ["S0", "S0", "S0", "S1", "S1", "S1"],
    "sink":   ["k0", "k1", "k2", "k0", "k1", "k2"],
    "cost":   [4, 3, 1, 5, 2, 3],
})
result = ortidy.transportation(
    edges, supply={"S0": 10, "S1": 15}, demand={"k0": 8, "k1": 9, "k2": 8}
)
print(result.status, "| total cost:", result.objective)
result.frame[result.frame["quantity"] > 0]

## Network flow

Max flow, min-cost flow, and shortest path over an edge list — each adds a flow column.

In [ ]:
edges = pd.DataFrame({"from": [0, 0, 1, 2], "to": [1, 2, 2, 3], "capacity": [3, 2, 2, 4]})
mf = ortidy.max_flow(edges, source=0, sink=3)
print("max flow:", mf.objective)
mf.frame

In [ ]:
edges = pd.DataFrame({
    "from": [0, 0, 1, 2], "to": [1, 2, 2, 3],
    "capacity": [10, 10, 10, 10], "cost": [4, 1, 1, 1],
})
supplies = pd.DataFrame({"node": [0, 3], "supply": [5, -5]})
mcf = ortidy.min_cost_flow(edges, supplies)
print("min cost:", mcf.objective)
mcf.frame

In [ ]:
edges = pd.DataFrame({"from": [0, 0, 1, 2], "to": [1, 2, 3, 3], "weight": [1, 4, 1, 1]})
sp = ortidy.shortest_path(edges, source=0, sink=3)
print("path length:", sp.objective)
sp.frame[sp.frame["onPath"] == 1]

## Facility location

`(customer, facility, cost)` edges plus per-facility setup cost. Opens facilities and assigns each customer to one.

In [ ]:
edges = pd.DataFrame({
    "customer": [0, 0, 1, 1, 2, 2],
    "facility": ["f0", "f1", "f0", "f1", "f0", "f1"],
    "cost":     [1, 9, 1, 9, 9, 1],
})
setup = pd.DataFrame({"facility": ["f0", "f1"], "setupCost": [2, 2]})
result = ortidy.facility_location(edges, setup)
print(result.status, "| total cost:", result.objective, "| opened:", result.metadata["opened"])
result.frame[result.frame["selected"]]

## Set cover

A `(subset, element)` membership list plus per-subset costs. Picks the cheapest subsets covering every element.

In [ ]:
membership = pd.DataFrame({
    "subset":  ["A", "A", "B", "B", "B", "C", "C"],
    "element": ["e0", "e1", "e1", "e2", "e3", "e0", "e2"],
})
costs = pd.DataFrame({"subset": ["A", "B", "C"], "cost": [3, 2, 2]})
result = ortidy.set_cover(membership, costs)
print(result.status, "| total cost:", result.objective)
result.frame

## Assignment with team caps

Group agents into teams and limit how many each team may use.

In [ ]:
edges = pd.DataFrame({
    "agent": ["a0", "a0", "a1", "a1", "a2", "a2", "a3", "a3"],
    "task":  ["t0", "t1", "t0", "t1", "t0", "t1", "t0", "t1"],
    "cost":  [1, 1, 1, 1, 9, 9, 9, 9],
})
teams = {"a0": "A", "a1": "A", "a2": "B", "a3": "B"}
result = ortidy.assignment(edges, teams=teams, team_capacity=1)
print(result.status, "| cost:", result.objective)
result.frame[result.frame["selected"]]

## Blending / diet (LP)

Choose continuous quantities of each item to meet requirements at least cost — the classic diet linear program.

In [ ]:
items = pd.DataFrame({
    "food":     ["bread", "milk", "cheese"],
    "cost":     [1.0, 2.0, 3.0],
    "protein":  [4.0, 8.0, 7.0],
    "calories": [90.0, 120.0, 100.0],
})
requirements = pd.DataFrame({"attribute": ["protein", "calories"], "min": [10.0, 150.0]})
result = ortidy.blend(items, requirements)
print(result.status, "| min cost:", round(result.objective, 2))
result.frame[["food", "quantity"]]